# Bayesian Inference - Examples

Practical demonstrations of Bayesian thinking for ML.

## Topics Covered
1. Bayes' Theorem Fundamentals
2. Beta-Binomial Conjugacy
3. Normal-Normal Conjugacy
4. Prior Influence and Prior Sensitivity
5. Sequential Updating
6. MAP vs MLE
7. Posterior Predictive Distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import beta as beta_func

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=6, suppress=True)

## 1. Bayes' Theorem Fundamentals

$$P(\theta|D) = \frac{P(D|\theta) \cdot P(\theta)}{P(D)}$$

- **Prior** $P(\theta)$: What we believe before seeing data
- **Likelihood** $P(D|\theta)$: How likely is data given parameter
- **Posterior** $P(\theta|D)$: Updated belief after seeing data
- **Evidence** $P(D)$: Normalizing constant

In [ ]:
print("BAYES' THEOREM: DIAGNOSTIC EXAMPLE")
print("="*60)

# Disease testing example
prior = 0.01  # 1% of population has disease
sensitivity = 0.95  # P(+|disease) = 95%
specificity = 0.90  # P(-|no disease) = 90%

print(f"\nSetup:")
print(f"  Prior P(disease) = {prior} (1%)")
print(f"  Sensitivity P(+|disease) = {sensitivity}")
print(f"  Specificity P(-|healthy) = {specificity}")

# Calculate P(disease|+)
p_positive_disease = sensitivity
p_positive_healthy = 1 - specificity
p_positive = p_positive_disease * prior + p_positive_healthy * (1 - prior)

posterior = (p_positive_disease * prior) / p_positive

print(f"\n📊 If test is POSITIVE:")
print(f"  P(+|disease) × P(disease) = {p_positive_disease} × {prior} = {p_positive_disease * prior:.4f}")
print(f"  P(+|healthy) × P(healthy) = {p_positive_healthy} × {1-prior} = {p_positive_healthy * (1-prior):.4f}")
print(f"  P(+) = {p_positive:.4f}")
print(f"")
print(f"  Posterior P(disease|+) = {posterior:.4f} ({posterior*100:.1f}%)")

print(f"\n💡 Key Insight:")
print(f"  Even with 95% sensitivity and 90% specificity,")
print(f"  a positive test only means {posterior*100:.1f}% chance of disease!")
print(f"  Low prior prevalence dominates the result.")

## 2. Beta-Binomial Conjugacy

**Prior**: $\theta \sim \text{Beta}(\alpha, \beta)$

**Data**: $k$ successes in $n$ trials

**Posterior**: $\theta | k \sim \text{Beta}(\alpha + k, \beta + n - k)$

In [ ]:
print("BETA-BINOMIAL CONJUGACY")
print("="*60)

# Scenario: Estimating click-through rate
# Prior: Beta(2, 8) - expect ~20% CTR
alpha_prior, beta_prior = 2, 8

# Data: 15 clicks in 50 views
clicks, views = 15, 50

# Posterior
alpha_post = alpha_prior + clicks
beta_post = beta_prior + (views - clicks)

print(f"\n📊 Prior: Beta({alpha_prior}, {beta_prior})")
print(f"  E[θ] = α/(α+β) = {alpha_prior/(alpha_prior+beta_prior):.4f}")

print(f"\n📊 Data: {clicks} clicks in {views} views")
print(f"  MLE: {clicks/views:.4f}")

print(f"\n📊 Posterior: Beta({alpha_post}, {beta_post})")
print(f"  E[θ|data] = {alpha_post/(alpha_post+beta_post):.4f}")

# 95% Credible Interval
ci = stats.beta.ppf([0.025, 0.975], alpha_post, beta_post)
print(f"  95% Credible Interval: [{ci[0]:.4f}, {ci[1]:.4f}]")

# Visualization
theta = np.linspace(0, 1, 200)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(theta, stats.beta.pdf(theta, alpha_prior, beta_prior), 'b--', lw=2, label=f'Prior Beta({alpha_prior},{beta_prior})')
ax.plot(theta, stats.beta.pdf(theta, clicks+1, views-clicks+1), 'g:', lw=2, label=f'Likelihood (scaled)')
ax.plot(theta, stats.beta.pdf(theta, alpha_post, beta_post), 'r-', lw=3, label=f'Posterior Beta({alpha_post},{beta_post})')
ax.axvline(clicks/views, color='g', linestyle=':', alpha=0.5, label=f'MLE={clicks/views:.3f}')
ax.axvspan(ci[0], ci[1], alpha=0.2, color='red', label='95% CI')
ax.set_xlabel('θ (CTR)')
ax.set_ylabel('Density')
ax.set_title('Beta-Binomial: Prior → Posterior')
ax.legend()
plt.show()

## 3. Normal-Normal Conjugacy

**Prior**: $\mu \sim N(\mu_0, \sigma_0^2)$

**Data**: $\bar{x}$ from $n$ samples with known $\sigma^2$

**Posterior**: Precision-weighted average of prior and data

In [ ]:
print("NORMAL-NORMAL CONJUGACY")
print("="*60)

# Prior belief about model accuracy
mu_prior = 0.75  # Expect 75%
sigma_prior = 0.10  # Uncertainty

# Data: sample mean of 0.82 from 10 runs
x_bar = 0.82
n = 10
sigma_data = 0.05  # Known population std

# Posterior (precision-weighted average)
precision_prior = 1 / sigma_prior**2
precision_data = n / sigma_data**2
precision_post = precision_prior + precision_data

mu_post = (precision_prior * mu_prior + precision_data * x_bar) / precision_post
sigma_post = np.sqrt(1 / precision_post)

print(f"\n📊 Prior: N({mu_prior}, {sigma_prior}²)")
print(f"  Precision: 1/σ² = {precision_prior:.2f}")

print(f"\n📊 Data: x̄ = {x_bar}, n = {n}, σ = {sigma_data}")
print(f"  Data precision: n/σ² = {precision_data:.2f}")

print(f"\n📊 Posterior: N({mu_post:.4f}, {sigma_post:.4f}²)")
print(f"  Posterior precision: {precision_post:.2f}")

# Credible interval
ci = stats.norm.ppf([0.025, 0.975], mu_post, sigma_post)
print(f"  95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")

# Weight analysis
weight_prior = precision_prior / precision_post
weight_data = precision_data / precision_post
print(f"\n💡 Weights:")
print(f"  Prior weight: {weight_prior:.2%}")
print(f"  Data weight: {weight_data:.2%}")

# Visualization
x = np.linspace(0.5, 1.0, 200)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(x, stats.norm.pdf(x, mu_prior, sigma_prior), 'b--', lw=2, label=f'Prior N({mu_prior}, {sigma_prior}²)')
ax.plot(x, stats.norm.pdf(x, x_bar, sigma_data/np.sqrt(n)), 'g:', lw=2, label=f'Likelihood (sampling dist)')
ax.plot(x, stats.norm.pdf(x, mu_post, sigma_post), 'r-', lw=3, label=f'Posterior N({mu_post:.3f}, {sigma_post:.3f}²)')
ax.axvline(mu_prior, color='b', linestyle='--', alpha=0.3)
ax.axvline(x_bar, color='g', linestyle=':', alpha=0.3)
ax.axvline(mu_post, color='r', linestyle='-', alpha=0.3)
ax.legend()
ax.set_xlabel('μ (accuracy)')
ax.set_ylabel('Density')
ax.set_title('Normal-Normal: Posterior is Between Prior and Data')
plt.show()

## 4. Prior Influence and Sensitivity

In [ ]:
print("PRIOR SENSITIVITY ANALYSIS")
print("="*60)

# Same data, different priors
clicks, views = 7, 20

priors = [
    (1, 1, "Uniform (uninformative)"),
    (2, 8, "Skeptical (expect 20%)"),
    (8, 2, "Optimistic (expect 80%)"),
    (20, 80, "Strong skeptical"),
]

print(f"\nData: {clicks}/{views} = {clicks/views:.1%} success rate")
print(f"\n{'Prior':<25} {'Posterior Mean':>15} {'95% CI':>25}")
print("-"*70)

theta = np.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(12, 6))

for alpha, beta, name in priors:
    alpha_post = alpha + clicks
    beta_post = beta + views - clicks
    mean_post = alpha_post / (alpha_post + beta_post)
    ci = stats.beta.ppf([0.025, 0.975], alpha_post, beta_post)
    
    print(f"{name:<25} {mean_post:>15.4f} [{ci[0]:.4f}, {ci[1]:.4f}]")
    
    ax.plot(theta, stats.beta.pdf(theta, alpha_post, beta_post), lw=2, 
            label=f'{name}: Post={mean_post:.3f}')

ax.axvline(clicks/views, color='black', linestyle='--', alpha=0.5, label=f'MLE={clicks/views:.3f}')
ax.legend()
ax.set_xlabel('θ')
ax.set_ylabel('Posterior Density')
ax.set_title('Prior Sensitivity: Same Data, Different Posteriors')
plt.show()

print("\n💡 With limited data (n=20), prior has strong influence.")
print("   Strong priors can overwhelm weak evidence!")

## 5. Sequential Updating

In [ ]:
print("SEQUENTIAL BAYESIAN UPDATING")
print("="*60)

np.random.seed(42)

# True parameter
true_theta = 0.65

# Start with uniform prior
alpha, beta = 1, 1

# Generate data in batches
batch_sizes = [5, 10, 20, 50, 100]

theta_range = np.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(14, 6))

# Plot prior
ax.plot(theta_range, stats.beta.pdf(theta_range, alpha, beta), 'k--', lw=1, label='Prior Beta(1,1)')

total_n = 0
colors = plt.cm.Reds(np.linspace(0.3, 1, len(batch_sizes)))

print(f"\n{'After n obs':>12} {'α':>6} {'β':>6} {'Post Mean':>12} {'95% CI':>20}")
print("-"*60)

for i, batch_size in enumerate(batch_sizes):
    # Generate new data
    new_data = np.random.binomial(1, true_theta, batch_size)
    successes = new_data.sum()
    
    # Update posterior
    alpha = alpha + successes
    beta = beta + batch_size - successes
    total_n += batch_size
    
    mean_post = alpha / (alpha + beta)
    ci = stats.beta.ppf([0.025, 0.975], alpha, beta)
    
    print(f"{total_n:>12} {alpha:>6} {beta:>6} {mean_post:>12.4f} [{ci[0]:.4f}, {ci[1]:.4f}]")
    
    ax.plot(theta_range, stats.beta.pdf(theta_range, alpha, beta), 
            color=colors[i], lw=2, label=f'n={total_n}')

ax.axvline(true_theta, color='green', linestyle='-', lw=2, alpha=0.7, label=f'True θ={true_theta}')
ax.legend(loc='upper left')
ax.set_xlabel('θ')
ax.set_ylabel('Density')
ax.set_title('Sequential Updating: Posterior Concentrates on True Value')
plt.show()

print(f"\n✅ As data accumulates, posterior converges to true θ = {true_theta}")

## 6. MAP vs MLE

In [ ]:
print("MAP vs MLE COMPARISON")
print("="*60)

# Scenario: Very limited data
successes, trials = 3, 4  # 75% observed

# Prior: Beta(2, 2) - slight preference for 50%
alpha_prior, beta_prior = 2, 2

# MLE
mle = successes / trials

# Posterior
alpha_post = alpha_prior + successes
beta_post = beta_prior + trials - successes

# MAP (mode of Beta distribution)
# Mode = (α-1)/(α+β-2) for α,β > 1
map_estimate = (alpha_post - 1) / (alpha_post + beta_post - 2)

# Posterior mean
post_mean = alpha_post / (alpha_post + beta_post)

print(f"\nData: {successes}/{trials} successes")
print(f"Prior: Beta({alpha_prior}, {beta_prior})")
print(f"Posterior: Beta({alpha_post}, {beta_post})")

print(f"\n📊 Point Estimates:")
print(f"  MLE: {mle:.4f} (purely data-driven)")
print(f"  MAP: {map_estimate:.4f} (mode of posterior)")
print(f"  Posterior Mean: {post_mean:.4f} (expected value)")

# Visualization
theta = np.linspace(0, 1, 200)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(theta, stats.beta.pdf(theta, alpha_prior, beta_prior), 'b--', lw=2, label='Prior')
ax.plot(theta, stats.beta.pdf(theta, alpha_post, beta_post), 'r-', lw=3, label='Posterior')
ax.axvline(mle, color='g', linestyle='-', lw=2, label=f'MLE={mle:.3f}')
ax.axvline(map_estimate, color='purple', linestyle='--', lw=2, label=f'MAP={map_estimate:.3f}')
ax.axvline(post_mean, color='orange', linestyle=':', lw=2, label=f'Post Mean={post_mean:.3f}')
ax.legend()
ax.set_xlabel('θ')
ax.set_ylabel('Density')
ax.set_title('MAP vs MLE: Prior Regularizes with Limited Data')
plt.show()

print("\n💡 Key Insight:")
print("  - MLE = 0.75 might overfit with only 4 observations")
print("  - MAP/Mean are shrunk toward prior, providing regularization")
print("  - This is the Bayesian foundation of regularization in ML!")

## 7. Posterior Predictive Distribution

In [ ]:
print("POSTERIOR PREDICTIVE DISTRIBUTION")
print("="*60)

# After observing data, predict future outcomes
alpha_post, beta_post = 12, 8  # Posterior from previous analysis

print(f"\nPosterior: Beta({alpha_post}, {beta_post})")
print(f"Posterior mean (expected θ): {alpha_post/(alpha_post+beta_post):.4f}")

# Predict: out of next 10 trials, how many successes?
n_future = 10

# Posterior predictive is Beta-Binomial
# P(k successes | data) = ∫ Binom(k|n,θ) × Beta(θ|α,β) dθ

k_values = np.arange(0, n_future + 1)
predictive_probs = []

for k in k_values:
    # Beta-Binomial PMF
    from scipy.special import comb
    prob = comb(n_future, k) * beta_func(alpha_post + k, beta_post + n_future - k) / beta_func(alpha_post, beta_post)
    predictive_probs.append(prob)

predictive_probs = np.array(predictive_probs)

# Compare with plug-in prediction (using posterior mean)
theta_hat = alpha_post / (alpha_post + beta_post)
plugin_probs = stats.binom.pmf(k_values, n_future, theta_hat)

print(f"\n{'k successes':>12} {'Predictive':>12} {'Plug-in':>12}")
print("-"*40)
for k, pred, plug in zip(k_values, predictive_probs, plugin_probs):
    print(f"{k:>12} {pred:>12.4f} {plug:>12.4f}")

# Visualization
fig, ax = plt.subplots(figsize=(12, 5))
width = 0.35
ax.bar(k_values - width/2, predictive_probs, width, label='Posterior Predictive', alpha=0.8)
ax.bar(k_values + width/2, plugin_probs, width, label='Plug-in (point estimate)', alpha=0.8)
ax.set_xlabel('Number of Successes (out of 10)')
ax.set_ylabel('Probability')
ax.set_title('Posterior Predictive vs Plug-in: Accounts for Uncertainty')
ax.legend()
ax.set_xticks(k_values)
plt.show()

print("\n💡 Posterior predictive distribution is WIDER")
print("   because it accounts for uncertainty in θ.")
print("   This is crucial for honest uncertainty quantification in ML!")